In [1]:
import geopandas as gpd
import pandas as pd
import geoplanar
import numpy as np

/home/eliska/extension2/.pixi/envs/default/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


source: https://ipi.eprostor.gov.si/jgp/data

In [2]:
nadzemni = gpd.read_file("/data/uscuni-ulce/extension2/slo/KN_SLO_STAVBE_SLO_nadzemni_tloris_20260809/KN_SLO_STAVBE_SLO_STAVBE_NADZEMNI_TLORIS_poligon.shp")

In [3]:
zps = gpd.read_file("/data/uscuni-ulce/extension2/slo/KN_SLO_STAVBE_SLO_tloris_zps_20260809/KN_SLO_STAVBE_SLO_STAVBE_TLORIS_ZPS_poligon.shp")

/home/eliska/extension2/.pixi/envs/default/lib/python3.13/site-packages/pyogrio/raw.py:200: RuntimeWarning: /data/uscuni-ulce/extension2/slo/KN_SLO_STAVBE_SLO_tloris_zps_20260809/KN_SLO_STAVBE_SLO_STAVBE_TLORIS_ZPS_poligon.shp contains polygon(s) with rings with invalid winding order. Autocorrecting them, but that shapefile should be corrected using ogr2ogr for example.
  return ogr_read(


In [ ]:
assert nadzemni.crs == zps.crs, "CRS mismatch — reproject one of them first"

# merge all nadzemni geometries into one, so a single vectorized 
# intersects() check tells you if ANY nadzemni polygon overlaps
nadzemni_union = nadzemni.geometry.unary_union

# boolean mask aligned 1:1 with zps — no index mismatch possible
has_overlap = zps.geometry.intersects(nadzemni_union)

zps_gap_fill = zps[~has_overlap].copy()

print(f"ZPS total: {len(zps)}")
print(f"ZPS with no nadzemni overlap (added as gap-fill): {len(zps_gap_fill)}")

nadzemni = nadzemni.copy()
nadzemni["footprint_source"] = "nadzemni"
zps_gap_fill["footprint_source"] = "zps_gapfill"
footprints = gpd.GeoDataFrame(
    pd.concat([nadzemni, zps_gap_fill], ignore_index=True),
    crs=nadzemni.crs
)

/tmp/ipykernel_3841276/2406430449.py:5: DeprecationWarning: The 'unary_union' attribute is deprecated, use the 'union_all()' method instead.
  nadzemni_union = nadzemni.geometry.unary_union


In [ ]:
#footprints.to_file("slo.gpkg")

In [12]:
footprints = footprints.to_crs(3035)

In [14]:
footprints = footprints.explode(index_parts=False).reset_index(drop=True)

In [16]:
footprints.to_parquet("/data/uscuni-ulce/extension2/slo/clean_3035.parquet")